# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E008-resnet-unified-finetune** — the recipe-and-input fix
after E006/E007's four identical unfreeze crashes. Three changes from E007:
(1) trainer fixes — GradScaler for fp16 autocast (gradients were silently
underflowing on the T4) and no weight decay on norms/biases; (2) input fix —
8 anchors over a (0.1, 0.9) window (~24 slices of coverage; the E005 2x2 showed
full-slice attention beats 9-slice triplets by a wide margin); (3) backbone —
resnet34, since fine-tuned CNNs are the community's 0.87 plateau and both DINOv2
fine-tunes crashed. Unified multi-plane model, cold start, tier weights, fixed
90/10 holdout. Success reads: frozen stage should approach ~0.77 (validating the
input fix), unfreeze should ADD, and >=0.80 clears the team's submission bar.

GPU cost: decode ~50 min + one fine-tune (~2x E007's epochs cost at 8 anchors,
~1.5h) — total ~2.5 T4 hours, single arm. Requires `WANDB_API_KEY`,
`knee-labels`, internet.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
COMMIT = "c8be415"  # main @ PR #23 squash: E008 trainer fixes + 8-anchor config
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; blended soft labels
# (with the per-cell __weight companions for E006a) via the knee-labels dataset;
# the E005 winner's feature bank + checkpoints via knee-e005-artifacts.
from pathlib import Path

from knee.data import load_blended_labels, weight_matrix

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)


def find_input(name: str, filename: str) -> Path:
    bases = [Path("/kaggle/input") / name, Path("/kaggle/input/datasets/josiemachalek") / name]
    for base in bases:
        if (base / filename).exists():
            return base / filename
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{filename} not found; mounts: {listing}")


labels = load_blended_labels(find_input("knee-labels", "blended_labels_v1.csv"), include_weights=True)
weights = weight_matrix(labels)
print(f"blended labels: {len(labels)} studies; weights {weights.shape}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E008 config. resnet34 (CNN robustness; DINOv2 crashed twice and gave nothing
# frozen at the LB), 8 anchors over a wide window (input fix), crop140 kept as
# harmless default, tier weights on. Cold start; frozen_epochs=4 gives the cold
# head+embeddings real convergence runway (E007's 2 were still climbing).
from knee.model import DEFAULT_BACKBONE

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DEFAULT_BACKBONE
INPUT_SIZE = 224
CROP_MM = 140.0
USE_WEIGHTS = True
E008_CONFIG = dict(n_anchors=8, anchor_window=(0.1, 0.9), frozen_epochs=4, epochs=18)

CHECKPOINT_DIR = Path("/kaggle/working")
CACHE_DIR = Path("/tmp/pixel_cache")  # ephemeral: only checkpoints persist as output

In [ ]:
# Fixed 90/10 split — the fine-tune-era regime marker. Same seed always, so every
# fine-tune era experiment shares the split and stays comparable.
import numpy as np

from knee.cv import stratified_holdout
from knee.labels import LABEL_COLUMNS

label_matrix = labels[list(LABEL_COLUMNS)].to_numpy(dtype=np.float32)
val_mask = stratified_holdout(label_matrix, val_fraction=0.1, seed=0)
print(f"split: {int((~val_mask).sum())} train / {int(val_mask.sum())} val")

In [ ]:
# E008: pixel cache (~50 min decode), then ONE unified fine-tune (~1.5h at 8
# anchors x up to 3 planes = up to 24 images per study). The 4 frozen warm-up
# epochs' printed val scores are the frozen baseline on this same split — they
# should approach the ~0.77-0.78 the full-slice frozen models earn if the input
# fix worked. The per-epoch val macro IS the holdout ensemble number.
from knee.finetune import FinetuneConfig, build_pixel_cache, finetune_unified
from knee.model import MultiPlaneModel

cache = build_pixel_cache(
    COMP_ROOT, labels, CACHE_DIR, series_types=SERIES_TYPES,
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
)
print("cache coverage:", {t.value: n for t, n in cache.coverage.items()})

model = MultiPlaneModel(BACKBONE, SERIES_TYPES)
result = finetune_unified(
    CACHE_DIR, label_matrix, val_mask,
    model=model,
    out_path=CHECKPOINT_DIR / "e008_unified.pt",
    config=FinetuneConfig(**E008_CONFIG),
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
    cell_weights=weights if USE_WEIGHTS else None,
    label_source=BLENDED_LABEL_SOURCE,
)
print(f"E008 unified: best val macro {result.best_val_macro_auc:.3f} (epoch {result.best_epoch + 1})")
print({label: round(auc, 3) for label, auc in result.val_auc_per_label.items()})

In [ ]:
# e008_unified.pt persists as notebook output. Submission bar (team policy
# 2026-09-03): >=0.80 holdout macro justifies the publish->infer->submit path;
# below that, record the numbers and bank the learning. If shipped, the single
# multiplane checkpoint REPLACES all knee-weights .pt files (inference
# auto-detects the kind; the plane-prior combiner retires with it).
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE, "crop_mm": CROP_MM,
        "tier_weighted": USE_WEIGHTS, "model_kind": "multiplane", **E008_CONFIG,
    })
    wandb.log({
        "holdout/unified_macro": result.best_val_macro_auc,
        **{f"holdout/unified/{label}": auc for label, auc in result.val_auc_per_label.items() if not math.isnan(auc)},
    })
    run.finish()